# Aula 06: montando o seu banco de fraude em cartão no FreeSQL

Este notebook cria as mesmas 6 tabelas usadas na aula, dentro do **seu** schema do FreeSQL, com os dados que a coordenação preparou. Rode as células na ordem, de cima para baixo.

**Antes de começar:**
- Tenha em mãos o DSN, o usuário e a senha do seu schema, do painel "Schema Connection Details" do FreeSQL (conforme apresentado pelo professor).
- Os arquivos `schema.sql` e a pasta `csv/` estão referenciados a partir das células desse notebook, você só precisa executar.

## 1. Instalar o driver do Oracle

In [ ]:
!pip install oracledb -q

## 2. Baixar os arquivos necessários

Em vez de montar o Google Drive, vamos baixar os arquivos necessários diretamente para o ambiente temporário do Colab.

In [ ]:
import gdown
import os

# Dicionário com IDs dos arquivos no Google Drive
files = {
    'schema.sql': '1VlfkmQWf6TVlwVT57pL57bRrfd5ekdJ1',
    'cartoes.csv': '1dvaJBzwQSuIZVMLyZR8UNm7tLRF9UD48',
    'categorias.csv': '12pB_IMoQvaKyq0Az2v5ZIVizs-OqKjPe',
    'clientes.csv': '1r7P8NFAt-51gJrVYWl-QBGOgvDL6-CVJ',
    'contestacoes.csv': '1KkAtzkJVQn_vSg1_FuroF_ZXwOxA4vmu',
    'lojistas.csv': '17BX38vXi69vK7MKxypyqNjkqBYHWjXOX',
    'transacoes.csv': '1Ec-LYcGsp-RVmEDrRcQ3K7BC5RWAJruJ'
}

# Criar pasta para os CSVs para manter a estrutura
os.makedirs('csv', exist_ok=True)

print("Baixando arquivos...")
for filename, file_id in files.items():
    url = f'https://drive.google.com/uc?id={file_id}'
    output = f'csv/{filename}' if filename.endswith('.csv') else filename
    gdown.download(url, output, quiet=True)
    print(f"- {filename} baixado.")

## 3. Definir caminhos locais

In [ ]:
PASTA_DADOS = "."

## 4. Conectar no seu FreeSQL

Preencha com os dados do painel "Schema Connection Details" da **sua** conta. A senha fica escondida enquanto você digita.

---

> **NOTA: VOCÊ PODE SUBSTITUIR NA SUA CÓPIA DESSE ARQUIVO A CÉLULA ABAIXO INTEIRA PELA VERSÃO QUE VOCÊ PEGOU NO FREESQL.**

Quando colar do FreeSQL, sua senha do banco FreeSQL ficará exposta aqui. Como é apenas um banco de dados sintético, temporário e de exemplo, não tem problemas!

**Em produção jamais exponha qualquer SECRET ou ENV_VAR.**

In [ ]:
import oracledb
import os
from dotenv import load_dotenv

load_dotenv()

local_dsn = "tcps://db.freesql.com:2484/26ai_un3c1"
connection = oracledb.connect(
    user="",
    password="",
    dsn=local_dsn)

print("Successfully connected to Oracle Database")

cursor = connection.cursor()
for result in cursor.execute("SELECT * FROM DUAL"):
    print(result)

## 5. Criar as 6 tabelas

Se você já rodou este notebook antes e quer recomeçar do zero, rode a célula de `DROP TABLE` logo abaixo primeiro. Se é a primeira vez, pule direto para a célula de `CREATE TABLE`.

In [ ]:
# Rode esta célula só se quiser apagar tabelas de uma tentativa anterior.
cursor = connection.cursor()
ordem_para_apagar = ['contestacoes', 'transacoes', 'cartoes', 'lojistas', 'clientes', 'categorias']
for tabela in ordem_para_apagar:
    try:
        cursor.execute(f'DROP TABLE {tabela}')
    except oracledb.DatabaseError as erro:
        codigo = erro.args[0].code
        if codigo != 942:  # 942 = tabela nao existe, pode ignorar
            raise
connection.commit()
print("Tabelas antigas removidas, se havia alguma.")

In [ ]:
import pathlib

# O arquivo schema.sql foi baixado na raiz
schema_sql = pathlib.Path("schema.sql").read_text(encoding="utf-8")
cursor = connection.cursor()
for instrucao in [s.strip() for s in schema_sql.split(';') if s.strip()]:
    cursor.execute(instrucao)
connection.commit()
print("As 6 tabelas foram criadas.")

## 6. Carregar os dados dos arquivos CSV

Cada tabela vira uma função que lê o CSV correspondente e insere em lotes de 2 mil linhas, na ordem certa por causa das chaves estrangeiras.

In [ ]:
import csv
from datetime import datetime

CONVERSORES = {
    "categorias": {"id_categoria": int},
    "clientes": {"id_cliente": int, "data_nascimento": "data",
                 "data_cadastro": "data", "score_risco": float},
    "lojistas": {"id_lojista": int, "id_categoria": int, "data_cadastro": "data"},
    "cartoes": {"id_cartao": int, "id_cliente": int, "data_emissao": "data",
                "limite": float},
    "transacoes": {"id_transacao": int, "id_cartao": int, "id_lojista": int,
                   "valor": float, "data_hora": "data"},
    "contestacoes": {"id_contestacao": int, "id_transacao": int,
                     "data_contestacao": "data"},
}

def converter_data(valor):
    try:
        return datetime.strptime(valor, "%Y-%m-%d %H:%M:%S")
    except ValueError:
        return datetime.strptime(valor, "%Y-%m-%d").date()

def converter_valor(valor, conversor):
    if valor == "":
        return None
    if conversor == "data":
        return converter_data(valor)
    if conversor in (int, float):
        return conversor(valor)
    return valor

def carregar_csv(nome_tabela, tamanho_lote=2000):
    # Ajustado para ler da subpasta 'csv' criada no download
    caminho = f"csv/{nome_tabela}.csv"
    conversores = CONVERSORES.get(nome_tabela, {})
    with open(caminho, encoding="utf-8") as f:
        leitor = csv.reader(f)
        colunas = next(leitor)
        linhas = []
        for linha_bruta in leitor:
            linha = [converter_valor(v, conversores.get(col))
                     for v, col in zip(linha_bruta, colunas)]
            linhas.append(tuple(linha))

    marcadores = ", ".join(f":{i + 1}" for i in range(len(colunas)))
    sql = f"INSERT INTO {nome_tabela} ({', '.join(colunas)}) VALUES ({marcadores})"
    total = len(linhas)
    for inicio in range(0, total, tamanho_lote):
        lote = linhas[inicio:inicio + tamanho_lote]
        cursor.executemany(sql, lote)
        connection.commit()
    print(f"{nome_tabela}: {total} linhas carregadas.")

In [ ]:
for tabela in ['categorias', 'clientes', 'lojistas', 'cartoes', 'transacoes', 'contestacoes']:
    carregar_csv(tabela)

## 7. Conferir se bateu
Resultado esperado:
```
categorias: 12 linhas
clientes: 1500 linhas
lojistas: 300 linhas
cartoes: 1971 linhas
transacoes: 50000 linhas
contestacoes: 1011 linhas
```

In [ ]:
for tabela in ['categorias', 'clientes', 'lojistas', 'cartoes', 'transacoes', 'contestacoes']:
    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    (total,) = cursor.fetchone()
    print(f'{tabela}: {total} linhas')

## Pronto!

O seu banco está populado, com o mesmo caso de fraude em cartão usado nesta Aula 06 do profesor, agora com 6 tabelas e volume de verdade. A partir daqui, use o worksheet do FreeSQL ou continue neste mesmo notebook para rodar as consultas SQL da aula, o que você se sentir mais confortável 😇.

## Responder as perguntas:
---
Aqui está o link para o arquivo .sql contendo as perguntas a serem respondidas hoje (27/08/2026):

[exercicios_aula_06_alunos.sql](https://drive.google.com/file/d/1FFvMx1HqNwTYBIbfdbvDodfZvGghODOa/view?usp=sharing)